# Chronos-Resonance Stream Dataset Capture

This Colab notebook captures GPU `clock64()` packets and writes them into a **stream-native replay dataset** instead of one giant object-array `.npy` file.

### Outputs
- `time_dilation_stream_dataset/manifest.json`
- `time_dilation_stream_dataset/chunk_*.npz`
- `time_dilation_stream_dataset.zip`

The layout is designed to be easier to replay as ordered moments for concurrent now-stream analysis and future live-style transport. Each emitted worker stream is written with an explicit logical `stream_id` so the learner can train one model per observed stream and then compare those models at the observer layer.

In [ ]:
# Install CuPy for GPU acceleration in Colab
!pip install cupy-cuda12x

In [ ]:
import cupy as cp

num_gpus = cp.cuda.runtime.getDeviceCount()
print(f"Detected {num_gpus} GPU(s) for Chronos-Resonance.")
for gpu_id in range(num_gpus):
    props = cp.cuda.runtime.getDeviceProperties(gpu_id)
    name = props['name']
    gpu_name = name.decode('utf-8') if isinstance(name, bytes) else str(name)
    print(f"GPU {gpu_id}: {gpu_name}")

In [ ]:
%%writefile workers.py
import json
import os
import queue as py_queue
import time
from pathlib import Path

import cupy as cp
import numpy as np

STREAM_DATASET_SCHEMA_VERSION = 1
STREAM_DATASET_TYPE = 'gpu_clock_time_stream_dataset'

cuda_code = r'''
extern "C" __global__
void record_time_jitter(unsigned long long* times, int num_samples) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    for (int i = 0; i < num_samples; i++) {
        asm volatile("mov.u64 %0, %%globaltimer;" : "=l"(times[tid * num_samples + i]));
    }
}
'''

def get_gpu_name(gpu_id):
    props = cp.cuda.runtime.getDeviceProperties(gpu_id)
    name = props['name']
    return name.decode() if isinstance(name, bytes) else str(name)

class ChunkedTimeStreamWriter:
    def __init__(self, dataset_dir, chunk_packet_target=256, workers_per_gpu=1):
        self.dataset_dir = Path(dataset_dir)
        self.dataset_dir.mkdir(parents=True, exist_ok=True)
        self.chunk_packet_target = max(int(chunk_packet_target), 1)
        self.workers_per_gpu = max(int(workers_per_gpu), 1)
        self.packet_buffer = []
        self.chunk_files = []
        self.packet_count = 0
        self.error_packets = []
        self.gpu_names = {}
        self.stream_manifests = {}
        self.host_time_start = None
        self.host_time_end = None

    def append(self, record):
        record_type = str(record.get('record_type', 'unknown'))
        if record_type == 'gpu_clock_packet':
            self.packet_buffer.append(record)
            self.packet_count += 1
            gpu_id = int(record.get('gpu_id', -1))
            stream_id = str(record.get('stream_id', f'gpu{gpu_id}_pid{int(record.get("pid", -1))}'))
            self.gpu_names[gpu_id] = str(record.get('gpu_name', 'unknown'))
            self.stream_manifests.setdefault(stream_id, {
                'stream_id': stream_id,
                'gpu_id': gpu_id,
                'gpu_name': str(record.get('gpu_name', 'unknown')),
                'pid': int(record.get('pid', -1)),
                'device_worker_index': int(record.get('device_worker_index', -1)),
                'stream_global_index': int(record.get('stream_global_index', -1)),
                'workers_per_gpu': max(int(record.get('workers_per_gpu', self.workers_per_gpu)), 1),
            })
            host_start = float(record.get('host_time_start', 0.0))
            host_end = float(record.get('host_time_end', host_start))
            self.host_time_start = host_start if self.host_time_start is None else min(self.host_time_start, host_start)
            self.host_time_end = host_end if self.host_time_end is None else max(self.host_time_end, host_end)
            if len(self.packet_buffer) >= self.chunk_packet_target:
                self.flush()
            return
        if record_type == 'worker_error':
            self.error_packets.append({
                'schema_version': int(record.get('schema_version', 1)),
                'record_type': 'worker_error',
                'gpu_id': int(record.get('gpu_id', -1)),
                'gpu_name': str(record.get('gpu_name', 'unknown')),
                'pid': int(record.get('pid', -1)),
                'stream_id': str(record.get('stream_id', 'unknown')),
                'device_worker_index': int(record.get('device_worker_index', -1)),
                'stream_global_index': int(record.get('stream_global_index', -1)),
                'host_time': float(record.get('host_time', time.time())),
                'error': str(record.get('error', 'unknown error')),
            })

    def flush(self):
        if not self.packet_buffer:
            return
        records = self.packet_buffer
        self.packet_buffer = []
        offsets = [0]
        flat_clock_parts = []
        for record in records:
            clock_data = np.asarray(record.get('clock_data', []), dtype=np.uint64).reshape(-1)
            flat_clock_parts.append(clock_data)
            offsets.append(offsets[-1] + int(len(clock_data)))
        flat_clock = np.concatenate(flat_clock_parts) if flat_clock_parts else np.empty(0, dtype=np.uint64)
        chunk_name = f'chunk_{len(self.chunk_files):05d}.npz'
        np.savez(
            self.dataset_dir / chunk_name,
            gpu_id=np.asarray([int(record.get('gpu_id', -1)) for record in records], dtype=np.int32),
            gpu_name=np.asarray([str(record.get('gpu_name', 'unknown')) for record in records]),
            pid=np.asarray([int(record.get('pid', -1)) for record in records], dtype=np.int64),
            stream_id=np.asarray([str(record.get('stream_id', 'unknown')) for record in records]),
            device_worker_index=np.asarray([int(record.get('device_worker_index', -1)) for record in records], dtype=np.int32),
            stream_global_index=np.asarray([int(record.get('stream_global_index', -1)) for record in records], dtype=np.int32),
            workers_per_gpu=np.asarray([max(int(record.get('workers_per_gpu', self.workers_per_gpu)), 1) for record in records], dtype=np.int32),
            packet_index=np.asarray([int(record.get('packet_index', -1)) for record in records], dtype=np.int64),
            host_time_start=np.asarray([float(record.get('host_time_start', 0.0)) for record in records], dtype=np.float64),
            host_time_end=np.asarray([float(record.get('host_time_end', 0.0)) for record in records], dtype=np.float64),
            host_time_mid=np.asarray([float(record.get('host_time_mid', 0.0)) for record in records], dtype=np.float64),
            elapsed_host_seconds=np.asarray([float(record.get('elapsed_host_seconds', 0.0)) for record in records], dtype=np.float64),
            threads_per_block=np.asarray([int(record.get('threads_per_block', 0)) for record in records], dtype=np.int32),
            blocks=np.asarray([int(record.get('blocks', 0)) for record in records], dtype=np.int32),
            samples_per_thread=np.asarray([int(record.get('samples_per_thread', 0)) for record in records], dtype=np.int32),
            total_threads=np.asarray([int(record.get('total_threads', 0)) for record in records], dtype=np.int32),
            buffer_size=np.asarray([int(record.get('buffer_size', 0)) for record in records], dtype=np.int32),
            sample_count=np.asarray([int(record.get('sample_count', 0)) for record in records], dtype=np.int32),
            clock_data_offset=np.asarray(offsets, dtype=np.int64),
            clock_data_flat=flat_clock,
        )
        self.chunk_files.append(chunk_name)

    def finalize(self):
        self.flush()
        manifest = {
            'schema_version': STREAM_DATASET_SCHEMA_VERSION,
            'dataset_type': STREAM_DATASET_TYPE,
            'created_unix_time': time.time(),
            'packet_count': int(self.packet_count),
            'chunk_packet_target': int(self.chunk_packet_target),
            'chunk_files': self.chunk_files,
            'gpu_count': int(len(self.gpu_names)),
            'gpus': [
                {'gpu_id': int(gpu_id), 'gpu_name': name}
                for gpu_id, name in sorted(self.gpu_names.items())
            ],
            'stream_count': int(len(self.stream_manifests)),
            'workers_per_gpu': int(self.workers_per_gpu),
            'streams': sorted(
                self.stream_manifests.values(),
                key=lambda item: (int(item.get('stream_global_index', -1)), str(item.get('stream_id', 'unknown'))),
            ),
            'error_count': int(len(self.error_packets)),
            'host_time_start': self.host_time_start,
            'host_time_end': self.host_time_end,
        }
        (self.dataset_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
        if self.error_packets:
            (self.dataset_dir / 'worker_errors.json').write_text(json.dumps(self.error_packets, indent=2), encoding='utf-8')
        return manifest

def gpu_worker(
    gpu_id,
    device_worker_index,
    stream_global_index,
    workers_per_gpu,
    queue,
    stop_event,
    threads_per_block=256,
    blocks=256,
    samples_per_thread=32,
    report_samples=8192,
):
    gpu_name = 'unknown'
    stream_id = f'gpu{int(gpu_id)}_worker{int(device_worker_index)}'
    try:
        cp.cuda.Device(gpu_id).use()
        gpu_name = get_gpu_name(gpu_id)
        
        # --- NEW: USE EXPLICIT CUDA STREAM FOR TRUE PARALLELISM ---
        # This prevents the GigaThread scheduler from serializing multiple workers
        # on the same GPU via the default stream (Stream.null).
        worker_stream = cp.cuda.Stream(non_blocking=True)
        
        record_kernel = cp.RawKernel(cuda_code, 'record_time_jitter')
        total_threads = threads_per_block * blocks
        buffer_size = total_threads * samples_per_thread
        sample_limit = min(buffer_size, report_samples)
        d_times = cp.empty(buffer_size, dtype=cp.uint64)
        packet_index = 0

        while not stop_event.is_set():
            host_time_start = time.time()
            
            # Record using the worker's private stream
            record_kernel((blocks,), (threads_per_block,), (d_times, samples_per_thread), stream=worker_stream)
            
            # Synchronize ONLY this stream to prevent serializing with other local workers
            # This is critical for maintaining high local entropy.
            worker_stream.synchronize()
            
            host_time_end = time.time()
            clock_data = cp.asnumpy(d_times[:sample_limit])
            
            queue.put({
                'schema_version': STREAM_DATASET_SCHEMA_VERSION,
                'record_type': 'gpu_clock_packet',
                'gpu_id': int(gpu_id),
                'gpu_name': gpu_name,
                'pid': int(os.getpid()),
                'stream_id': stream_id,
                'device_worker_index': int(device_worker_index),
                'stream_global_index': int(stream_global_index),
                'workers_per_gpu': max(int(workers_per_gpu), 1),
                'packet_index': int(packet_index),
                'host_time_start': float(host_time_start),
                'host_time_end': float(host_time_end),
                'host_time_mid': float((host_time_start + host_time_end) / 2.0),
                'elapsed_host_seconds': float(host_time_end - host_time_start),
                'threads_per_block': int(threads_per_block),
                'blocks': int(blocks),
                'samples_per_thread': int(samples_per_thread),
                'total_threads': int(total_threads),
                'buffer_size': int(buffer_size),
                'sample_count': int(len(clock_data)),
                'clock_data': clock_data,
            })
            packet_index += 1
    except Exception as exc:
        queue.put({
            'schema_version': STREAM_DATASET_SCHEMA_VERSION,
            'record_type': 'worker_error',
            'gpu_id': int(gpu_id),
            'gpu_name': gpu_name,
            'pid': int(os.getpid()),
            'stream_id': stream_id,
            'device_worker_index': int(device_worker_index),
            'stream_global_index': int(stream_global_index),
            'host_time': float(time.time()),
            'error': repr(exc),
        })
        print(f'[GPU {gpu_id} worker {device_worker_index}] Error: {exc}')

def receiver_worker(queue, stop_event, dataset_dir='time_dilation_stream_dataset', chunk_packet_target=256, progress_interval=1024, workers_per_gpu=1):
    writer = ChunkedTimeStreamWriter(dataset_dir, chunk_packet_target=chunk_packet_target, workers_per_gpu=workers_per_gpu)
    empty_polls_after_stop = 0
    while True:
        try:
            record = queue.get(timeout=0.5)
            empty_polls_after_stop = 0
        except py_queue.Empty:
            if stop_event.is_set():
                empty_polls_after_stop += 1
                if empty_polls_after_stop >= 4:
                    break
            continue
        writer.append(record)
        if writer.packet_count and writer.packet_count % int(progress_interval) == 0:
            print(f'[receiver] captured {writer.packet_count} packets so far across {len(writer.stream_manifests)} streams')
    manifest = writer.finalize()
    print(json.dumps({
        'dataset_dir': str(dataset_dir),
        'packet_count': int(manifest['packet_count']),
        'chunk_count': int(len(manifest['chunk_files'])),
        'stream_count': int(manifest.get('stream_count', 0)),
        'workers_per_gpu': int(manifest.get('workers_per_gpu', 1)),
        'error_count': int(manifest['error_count']),
    }, indent=2))

In [ ]:
import json
import multiprocessing as mp
import shutil
import time
from pathlib import Path

import cupy as cp
import workers

DATASET_DIR = 'time_dilation_stream_dataset'
DURATION_SECONDS = 256
CHUNK_PACKET_TARGET = 256
WORKERS_PER_GPU = 4
QUEUE_MAXSIZE = 256

if __name__ == '__main__':
    dataset_path = Path(DATASET_DIR)
    if dataset_path.exists():
        shutil.rmtree(dataset_path)

    ctx = mp.get_context('spawn')
    queue = ctx.Queue(maxsize=QUEUE_MAXSIZE)
    stop_event = ctx.Event()

    num_gpus = cp.cuda.runtime.getDeviceCount()
    if num_gpus == 0:
        raise RuntimeError('No GPUs are visible to CuPy in this runtime.')
    total_streams = int(num_gpus * max(int(WORKERS_PER_GPU), 1))

    receiver = ctx.Process(
        target=workers.receiver_worker,
        args=(queue, stop_event, DATASET_DIR, CHUNK_PACKET_TARGET, 1024, WORKERS_PER_GPU),
    )
    receiver.start()

    emitters = []
    for gpu_id in range(num_gpus):
        for device_worker_index in range(max(int(WORKERS_PER_GPU), 1)):
            stream_global_index = gpu_id * max(int(WORKERS_PER_GPU), 1) + device_worker_index
            process = ctx.Process(
                target=workers.gpu_worker,
                args=(gpu_id, device_worker_index, stream_global_index, WORKERS_PER_GPU, queue, stop_event),
            )
            process.start()
            emitters.append(process)

    print(f'Stream capture running for {DURATION_SECONDS} seconds across {num_gpus} visible GPU(s) and {total_streams} observed worker streams...')
    time.sleep(DURATION_SECONDS)
    stop_event.set()

    for process in emitters:
        process.join(timeout=30)
        if process.is_alive():
            process.terminate()

    receiver.join(timeout=120)
    if receiver.is_alive():
        receiver.terminate()

    manifest = json.loads((dataset_path / 'manifest.json').read_text(encoding='utf-8'))
    print(json.dumps({
        'dataset_dir': DATASET_DIR,
        'packet_count': int(manifest['packet_count']),
        'chunk_count': int(len(manifest['chunk_files'])),
        'gpu_count': int(manifest['gpu_count']),
        'stream_count': int(manifest.get('stream_count', 0)),
        'workers_per_gpu': int(manifest.get('workers_per_gpu', 1)),
        'error_count': int(manifest['error_count']),
    }, indent=2))

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

dataset_dir = Path('time_dilation_stream_dataset')
manifest = json.loads((dataset_dir / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, indent=2))

if manifest['chunk_files']:
    first_chunk = dataset_dir / manifest['chunk_files'][0]
    with np.load(first_chunk, allow_pickle=False) as chunk:
        offsets = np.asarray(chunk['clock_data_offset'], dtype=np.int64)
        flat_clock = np.asarray(chunk['clock_data_flat'], dtype=np.uint64)
        if len(offsets) > 1:
            first_packet = flat_clock[int(offsets[0]):int(offsets[1])]
            preview = first_packet[: min(len(first_packet), 1024)].astype(np.int64)
            if len(preview) > 1:
                deltas = np.diff(preview)
                plt.figure(figsize=(12, 4))
                plt.plot(deltas[:256], linewidth=1)
                plt.title('First packet local clock delta preview')
                plt.xlabel('Sample index')
                plt.ylabel('clock64 delta')
                plt.grid(True, alpha=0.3)
                plt.show()
        stream_ids = np.asarray(chunk['stream_id']).astype(str) if 'stream_id' in chunk.files else np.asarray([], dtype=str)
        print({
            'first_chunk': str(first_chunk),
            'packet_rows_in_chunk': int(len(chunk['gpu_id'])),
            'flat_clock_samples_in_chunk': int(len(flat_clock)),
            'stream_ids_in_first_chunk': sorted({str(value) for value in stream_ids.tolist()})[:8],
        })

In [ ]:
import shutil

from google.colab import files

dataset_dir = 'time_dilation_stream_dataset'
archive_path = shutil.make_archive(dataset_dir, 'zip', root_dir='.', base_dir=dataset_dir)
print(f'Created archive: {archive_path}')
files.download(archive_path)